# V4 DINOv3 7B teacher memory profile

This Kaggle-only gate inspects the pinned DINOv3 7B SAT-493M artifact and the actual accelerator before attempting any model allocation. It writes a structured report for the next teacher decision. A conservative memory check prevents an avoidable out-of-memory failure; the 7B model is instantiated and run on one real xBD pair only when the measured device has sufficient headroom.


In [ ]:
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_PATH = Path("/kaggle/working/rv4_dinov3_teacher_profile.json")
MODEL_ID = "facebook/dinov3-vit7b16-pretrain-sat493m"
MODEL_ENTRYPOINT = "dinov3_vit7b16"
MODEL_ARTIFACT = "dinov3_vit7b16_pretrain_sat493m"
ATTACHED_SOURCE_ROLE = "LVD1689M_7B_size_proxy_not_SAT493M"
EXPECTED_PARAMETER_COUNT = 7_000_000_000
PATCH_SIZE = 16
PROFILE_SIZE = 224


def candidate_model_files() -> list[Path]:
    patterns = ("*.pth", "*.pt", "*.bin", "*.safetensors", "*.keras", "*.h5")
    files = []
    for pattern in patterns:
        files.extend(INPUT_ROOT.rglob(pattern))
    return sorted({path for path in files if path.is_file()})


def first_pair() -> tuple[Path, Path]:
    pre_paths = sorted(INPUT_ROOT.rglob("*_pre_disaster.png"))
    if not pre_paths:
        raise RuntimeError("the attached xBD dataset has no pre-disaster image")
    pre_path = pre_paths[0]
    post_path = pre_path.with_name(pre_path.name.replace("_pre_disaster.png", "_post_disaster.png"))
    if not post_path.exists():
        raise RuntimeError(f"paired post-disaster image is missing: {post_path}")
    return pre_path, post_path


def load_rgb(path: Path) -> torch.Tensor:
    with Image.open(path) as image:
        array = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
    array = array[:PROFILE_SIZE, :PROFILE_SIZE, :]
    if array.shape != (PROFILE_SIZE, PROFILE_SIZE, 3):
        raise RuntimeError(f"unexpected profile crop shape for {path}: {array.shape}")
    return torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0)


def extract_patch_tokens(output):
    if isinstance(output, dict):
        for key in ("x_norm_patchtokens", "patch_tokens", "last_hidden_state"):
            value = output.get(key)
            if torch.is_tensor(value):
                return value, key
    for key in ("x_norm_patchtokens", "patch_tokens", "last_hidden_state"):
        value = getattr(output, key, None)
        if torch.is_tensor(value):
            return value, key
    if torch.is_tensor(output):
        return output, "tensor"
    raise RuntimeError(f"unsupported DINOv3 feature output type: {type(output).__name__}")


def write_report(report: dict) -> None:
    OUTPUT_PATH.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
    print(json.dumps(report, indent=2, sort_keys=True))


def main() -> None:
    if not torch.cuda.is_available():
        write_report({
            "schema_version": "rv4-dinov3-7b-teacher-profile-1",
            "status": "BLOCKED_NO_CUDA",
            "model_id": MODEL_ID,
            "model_entrypoint": MODEL_ENTRYPOINT,
            "attached_source_role": ATTACHED_SOURCE_ROLE,
        })
        return

    device = torch.device("cuda")
    free_bytes, total_bytes = torch.cuda.mem_get_info(device)
    model_files = candidate_model_files()
    file_records = [
        {"path": str(path), "bytes": path.stat().st_size}
        for path in model_files
    ]
    if not file_records:
        write_report({
            "schema_version": "rv4-dinov3-7b-teacher-profile-1",
            "status": "BLOCKED_MODEL_ARTIFACT_MISSING",
            "model_id": MODEL_ID,
            "model_entrypoint": MODEL_ENTRYPOINT,
            "attached_source_role": ATTACHED_SOURCE_ROLE,
            "gpu_name": torch.cuda.get_device_name(device),
            "free_memory_bytes": int(free_bytes),
            "total_memory_bytes": int(total_bytes),
            "model_files": [],
        })
        return

    # A 7B fp16/bf16 checkpoint needs roughly 14 GB for weights alone. The
    # extra factor and fixed margin cover model buffers, CUDA allocator
    # fragmentation and one modest feature forward; this is deliberately
    # conservative so the gate never turns a known OOM into a failed run.
    weight_bytes = EXPECTED_PARAMETER_COUNT * 2
    projected_runtime_bytes = int(weight_bytes * 1.35 + 4 * 1024**3)
    memory_fits = projected_runtime_bytes < free_bytes
    report = {
        "schema_version": "rv4-dinov3-7b-teacher-profile-1",
        "model_id": MODEL_ID,
        "model_entrypoint": MODEL_ENTRYPOINT,
        "model_artifact": MODEL_ARTIFACT,
        "attached_source_role": ATTACHED_SOURCE_ROLE,
        "expected_parameter_count": EXPECTED_PARAMETER_COUNT,
        "gpu_name": torch.cuda.get_device_name(device),
        "torch_version": torch.__version__,
        "free_memory_bytes_before_load": int(free_bytes),
        "total_memory_bytes": int(total_bytes),
        "weight_bytes_fp16_or_bf16_estimate": int(weight_bytes),
        "projected_runtime_bytes": projected_runtime_bytes,
        "model_files": file_records,
        "memory_gate_fits": bool(memory_fits),
    }
    if not memory_fits:
        report.update({
            "status": "PROFILED_INSUFFICIENT_MEMORY",
            "load_attempted": False,
            "recommendation": "Use an A100-class or larger GPU, or run an explicitly validated teacher compression/offload experiment before feature extraction.",
        })
        write_report(report)
        return

    hubconf_paths = sorted(INPUT_ROOT.rglob("hubconf.py"))
    checkpoint_paths = [path for path in model_files if path.suffix in {".pth", ".pt", ".bin"}]
    if not hubconf_paths or not checkpoint_paths:
        report.update({
            "status": "BLOCKED_LOADER_ARTIFACT_MISSING",
            "load_attempted": False,
            "hubconf_paths": [str(path) for path in hubconf_paths],
            "checkpoint_paths": [str(path) for path in checkpoint_paths],
        })
        write_report(report)
        return

    repo_root = hubconf_paths[0].parent
    import sys
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))
    weights = checkpoint_paths[0]
    torch.cuda.reset_peak_memory_stats(device)
    started = time.perf_counter()
    model = torch.hub.load(str(repo_root), MODEL_ENTRYPOINT, source="local", weights=str(weights))
    model = model.to(device).eval()
    load_seconds = time.perf_counter() - started
    pre_path, post_path = first_pair()
    pre = load_rgb(pre_path).to(device)
    post = load_rgb(post_path).to(device)
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=dtype):
        torch.cuda.synchronize(device)
        infer_started = time.perf_counter()
        pre_output = model.forward_features(pre) if hasattr(model, "forward_features") else model(pre)
        post_output = model.forward_features(post) if hasattr(model, "forward_features") else model(post)
        torch.cuda.synchronize(device)
        infer_seconds = time.perf_counter() - infer_started
    pre_tokens, pre_key = extract_patch_tokens(pre_output)
    post_tokens, post_key = extract_patch_tokens(post_output)
    expected_tokens = (PROFILE_SIZE // PATCH_SIZE) ** 2
    if pre_tokens.ndim != 3 or post_tokens.ndim != 3 or pre_tokens.shape != post_tokens.shape:
        raise RuntimeError(f"unexpected teacher token shapes: {tuple(pre_tokens.shape)} and {tuple(post_tokens.shape)}")
    if pre_tokens.shape[1] != expected_tokens:
        raise RuntimeError(f"expected {expected_tokens} spatial tokens, got {pre_tokens.shape[1]}")
    report.update({
        "status": "PASS",
        "load_attempted": True,
        "load_seconds": load_seconds,
        "paired_inference_seconds": infer_seconds,
        "dtype": str(dtype),
        "pair": {"pre": str(pre_path), "post": str(post_path)},
        "input_shape": list(pre.shape),
        "pre_output_key": pre_key,
        "post_output_key": post_key,
        "token_shape": list(pre_tokens.shape),
        "expected_spatial_tokens": expected_tokens,
        "embedding_dim": int(pre_tokens.shape[-1]),
        "model_parameter_count": sum(parameter.numel() for parameter in model.parameters()),
        "peak_memory_allocated": int(torch.cuda.max_memory_allocated(device)),
        "peak_memory_reserved": int(torch.cuda.max_memory_reserved(device)),
    })
    write_report(report)


if __name__ == "__main__":
    main()
